# Study 814 — Trailing-Sharpe Anomaly — the teardown

The per-leg splits, the Newey-West spread *t*, the Sharpe-vs-momentum-vs-lowvol head-to-head and rank overlap, the 1,000-permutation placebo, the two-era cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3873, 'spread_bps': 1.29, 't_nw': 0.83, 't_1s': 0.79, 'hi_bps': 8.05, 'lo_bps': 6.75, 'welch_t': 0.46, 'gross_sharpe': 0.2, 'mom_bps': 1.59, 'mom_t': 0.99, 'lowvol_bps': -5.65, 'lowvol_t': -2.92, 'rho_sharpe_mom': 0.953, 'rho_sharpe_negvol': 0.088, 'placebo_obs': 1.29, 'placebo_mean': 0.057, 'placebo_sd': 0.986, 'placebo_p': 0.096, 'placebo_sigma': 1.25, 'placebo_draws': 1000, 'era_early_bps': 0.28, 'era_early_t': 0.15, 'era_early_n': 1739, 'era_late_bps': 2.12, 'era_late_t': 0.89, 'era_late_n': 2134, 'timer_1_gross': 1.29, 'timer_1_cost': 2.14, 'timer_1_net': -0.84, 'timer_1_t': -0.51, 'timer_5_gross': 1.29, 'timer_5_cost': 10.14, 'timer_5_net': -8.84, 'timer_5_t': -5.4, 'null_mean_t': -0.03, 'null_sd_t': 0.93, 'null_fire': 1, 'planted_t': 6.64, 'planted_welch': 6.62}

## The headline — long-high-Sharpe / short-low-Sharpe spread

Daily equal-weight top-30% minus bottom-30% trailing-Sharpe spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-Sharpe {R['hi_bps']:+.2f} vs low-Sharpe {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +1.29 bps/day  NW(10) t = +0.83  one-sample t = +0.79
books         : high-Sharpe +8.05 vs low-Sharpe +6.75 bps (Welch t = +0.46)
gross Sharpe  : 0.20 (before cost)


## Does risk-adjusting help? — the head-to-head (the whole point)

Same universe, same dates, same sort machinery — Sharpe vs plain momentum vs pure low-vol, plus the average per-day rank overlap of the signals.

In [3]:
print(f"trailing Sharpe : {R['spread_bps']:+.2f} bps  NW t = {R['t_nw']:+.2f}")
print(f"12-1 momentum   : {R['mom_bps']:+.2f} bps  NW t = {R['mom_t']:+.2f}")
print(f"pure low-vol    : {R['lowvol_bps']:+.2f} bps  NW t = {R['lowvol_t']:+.2f}")
print(f"rank corr  Sharpe~momentum = {R['rho_sharpe_mom']:+.3f}  "
      f"Sharpe~(-vol) = {R['rho_sharpe_negvol']:+.3f}")
print('=> 0.95 correlated with momentum, ~0 low-vol content: momentum repackaged.')

trailing Sharpe : +1.29 bps  NW t = +0.83
12-1 momentum   : +1.59 bps  NW t = +0.99
pure low-vol    : -5.65 bps  NW t = -2.92
rank corr  Sharpe~momentum = +0.953  Sharpe~(-vol) = +0.088
=> 0.95 correlated with momentum, ~0 low-vol content: momentum repackaged.


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}  (~{R['placebo_sigma']:.2f} sigma)")

observed +1.29 bps vs placebo mean +0.057 (sd 0.986) -> p = 0.09600  (~1.25 sigma)


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1739): +0.28 bps  NW t = +0.15
2018-2026 (n=2134): +2.12 bps  NW t = +0.89


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +1.29 -> net -0.84 bps/day (cost 2.14/day, t=-0.51)
5 bps one-way: gross +1.29 -> net -8.84 bps/day (cost 10.14/day, t=-5.40)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from trailing_sharpe import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=814+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=814, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.20 (sd 0.78), |t|>=2 in 0/8


planted (edge=0.0016): NW t = +6.64, Welch t = +6.62


## Verdict

- **Signal — None.** The trailing-Sharpe sort earns **+1.29 bps/day** (NW *t* = **+0.83**) — right sign, not significant. At **0.95** rank correlation with plain 12-1 momentum and only +0.09 with low-vol, it is momentum repackaged and earns a touch *less* than the momentum book itself (+1.59 bps, *t* +0.99); neither clears |t| ≥ 2, it sits ~1.2σ into the placebo, and it is flat in both eras (*t* = +0.15 / +0.89). The synthetic control recovers a *planted* effect (*t* = +6.64, fires on 1/20 nulls ≈ nominal 5%), so the flat real-tape read is genuine. Survivorship biases the magnitude upward.
- **Tradability — Mirage.** The insignificant gross edge goes net-negative at 1 bp one-way (-0.84 bps/day, *t* = -0.51) as the 2.14 bps/day friction eats it; at 5 bps **-8.84 bps/day**.